# 03 — Visualizations

This notebook contains visualization templates (skeletons).  
Real data will be plugged in after processing is finalized.

In [ ]:
import sys
sys.path.append("..")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from config import *
from src.plot_utils import set_style, save_fig

set_style()

**Chart 1: MyChart activation rate by age group (10-year bins)**
What we include:
Include only real status: Activated, Inactive, Pending Activation, Patient Declined, Activation Code Generated but Disabled, and Non Standard MyChart Status.

We exclude *Unspecified because it means the MyChart status is unknown.This would lead to lower activation rate because number of *Unspecified response is huge

Calculation for activation rate:
Activated = 1, every other known status = 0.
Then take the average within each 10-year age group.

In [ ]:
# --- MyChart activation rate by age group ---

from src.data_loader import load_patients

# --- MyChart Activation Rate by Age Group ---

# Load patients dataset
pat = load_patients()

# Compute age from birth year
pat["age"] = 2026 - pat["PatientBirthYearBin"]

# Create 10-year age bins (0–9, 10–19, ..., 90–99)
bins = np.arange(0, 101, 10)
labels = [f"{i}-{i+9}" for i in bins[:-1]]

pat["age_group"] = pd.cut(
    pat["age"],
    bins=bins,
    labels=labels,
    right=False
)

# Exclude unknown MyChart status to avoid misclassification
pat = pat[pat["MyChartStatus"] != "*Unspecified"].copy()

# Convert MyChart status to binary:
# Activated → 1, all other known statuses → 0
pat["is_activated"] = pat["MyChartStatus"] == "Activated"

# Compute activation rate per age group
activation_df = (
    pat.groupby("age_group", observed=True)["is_activated"]
    .mean()
    .reset_index(name="activation_rate")
)

# Plot activation rate
plt.figure(figsize=(10,6))
sns.barplot(data=activation_df, x="age_group", y="activation_rate")

plt.title("MyChart Activation Rate by Age Group")
plt.xlabel("Age Group")
plt.ylabel("Activation Rate")

save_fig(plt.gcf(), "mychart_activation_by_age_group")
plt.show()

**Chart 2: ED Rate: App Users vs Non-App Users**
Each encounter represents a single visit, with IsEdVisit indicating whether the visit was an emergency (1) or not (0). The ED rate is calculated as the proportion of encounters that are ED visits within each group.

In [ ]:
# --- ED rate: app users vs non-app users ---

from src.data_loader import load_patients, load_encounters

# Load full datasets
pat = load_patients()

enc = load_encounters(
    cols=["PatientDurableKey", "IsEdVisit"],
    sample_frac=1.0   # FULL DATA
)

# Exclude unknown MyChart status
pat_mychart = pat[pat["MyChartStatus"] != "*Unspecified"].copy()

# Define app user vs non-app user
pat_mychart["app_status"] = np.where(
    pat_mychart["MyChartStatus"] == "Activated",
    "App User",
    "Non-App User"
)

# Keep only needed columns
pat_status = pat_mychart[["DurableKey", "app_status"]]

# Merge encounters with patient app status
enc_app = enc.merge(
    pat_status,
    left_on="PatientDurableKey",
    right_on="DurableKey",
    how="inner"
)

# Compute ED rate
# mean(IsEdVisit) = (# ED visits) / (total visits)
ed_rate_df = (
    enc_app
    .groupby("app_status")["IsEdVisit"]
    .mean()
    .reset_index(name="ed_rate")
)

# Plot
plt.figure(figsize=(7, 5))

sns.barplot(
    data=ed_rate_df,
    x="app_status",
    y="ed_rate"
)

plt.title("ED Rate: App Users vs Non-App Users")
plt.xlabel("App Status")
plt.ylabel("ED Visit Rate")

save_fig(plt.gcf(), "ed_rate_app_vs_nonapp")
plt.show()

**Chart 3: Journey length distribution by app status**
Journey length of each patient here is calculated as date of last visit - date of first visit (duration, NOT number of visit)
We compare the duration of visits between two groups of patients: App Users ( "Activated" status on Mychart and Non-app Users (all other statuses))

In [ ]:
# --- Journey Length Distribution by App Status ---
from src.data_loader import load_patients, load_encounters
# Load data
pat = load_patients()
enc = load_encounters(cols=["PatientDurableKey", "Date"], sample_frac=1.0)

# Convert Date column to datetime
enc["Date"] = pd.to_datetime(enc["Date"], errors="coerce")

# Remove invalid/missing dates
enc = enc.dropna(subset=["Date"])

# Compute journey length per patient:
# first visit ->  last visit
journey = (
    enc.groupby("PatientDurableKey")["Date"]
    .agg(first_visit="min", last_visit="max")
    .reset_index()
)

# Duration in days
journey["journey_length_days"] = (
    journey["last_visit"] - journey["first_visit"]
).dt.days

# Exclude unknown MyChart status
pat = pat[pat["MyChartStatus"] != "*Unspecified"].copy()

# Define app status
pat["app_status"] = np.where(
    pat["MyChartStatus"] == "Activated",
    "App User",
    "Non-App User"
)

# Merge journey with patient app status
journey = journey.merge(
    pat[["DurableKey", "app_status"]],
    left_on="PatientDurableKey",
    right_on="DurableKey",
    how="inner"
)

# Plot distribution (boxplot)
plt.figure(figsize=(8,5))
sns.boxplot(data=journey, x="app_status", y="journey_length_days")

plt.title("Journey Length Distribution by App Status")
plt.xlabel("App Status")
plt.ylabel("Journey Length (Days)")

save_fig(plt.gcf(), "journey_length_distribution")
plt.show()

**Chart 4: Follow up gap distribution (How long does it take patient to have a follow up after their appointment)**
Defined as next follow-up visit - appointment time for the same patient. 
Distribution is compared between App users and Non-app users, excluding unspecified status patients.

In [ ]:
# --- Follow-up Gap Distribution by App Status ---

from src.data_loader import load_patients, load_encounters

# Load data
pat = load_patients()
enc = load_encounters(cols=["PatientDurableKey", "Date", "Type"], sample_frac=1.0)

# Convert Date to datetime
enc["Date"] = pd.to_datetime(enc["Date"], errors="coerce")

# Keep only Appointment and Follow-Up visits
enc = enc[enc["Type"].isin(["Appointment", "Follow-Up"])].copy()

# Sort by patient and date
enc = enc.sort_values(["PatientDurableKey", "Date"])

# Get next visit type/date for each patient
enc["next_type"] = enc.groupby("PatientDurableKey")["Type"].shift(-1)
enc["next_date"] = enc.groupby("PatientDurableKey")["Date"].shift(-1)

# Keep only Appointment → Follow-Up pairs
gaps = enc[
    (enc["Type"] == "Appointment") &
    (enc["next_type"] == "Follow-Up")
].copy()

# Compute gap in days
gaps["followup_gap_days"] = (gaps["next_date"] - gaps["Date"]).dt.days

# Remove invalid (negative) gaps
gaps = gaps[gaps["followup_gap_days"] >= 0]

# Exclude unknown MyChart status
pat = pat[pat["MyChartStatus"] != "*Unspecified"].copy()

# Define app status
pat["app_status"] = np.where(
    pat["MyChartStatus"] == "Activated",
    "App User",
    "Non-App User"
)

# Merge with patient info
gaps = gaps.merge(
    pat[["DurableKey", "app_status"]],
    left_on="PatientDurableKey",
    right_on="DurableKey",
    how="inner"
)

# Plot violin distribution
plt.figure(figsize=(8,5))
sns.violinplot(
    data=gaps,
    x="app_status",
    y="followup_gap_days",
    inner="quartile",
    cut=0
)

plt.title("Follow-up Gap Distribution by App Status")
plt.xlabel("App Status")
plt.ylabel("Follow-up Gap (Days)")

save_fig(plt.gcf(), "followup_gap_distribution")
plt.show()

**Chart 5: Distance vs ED rate**
Scatter plot that show relationship between geographic distance from healthcare services vs reliance on ED visits. Aim to identify patients that liuve further from care centers are mor elikely to use emergency services due to limited access to routine care
Link each patients CensusBlockGroupFipsCode to centroid coordinates (CENTLAT, CENTLON from tigercensuscodes dataset)
Use 2 functions from distance_utils.py: attach_patient_coords() and 
haversine_km()

In [ ]:
# --- Distance vs ED Rate ---

# Goal:
# Each point represents a census block group.
# X-axis = average distance from the hospital/reference point
# Y-axis = ED visit rate for patients in that census block group

from src.data_loader import load_patients, load_encounters, load_tiger
from src.distance_utils import haversine_km, attach_patient_coords

# --- Load needed datasets ---
pat = load_patients()
tiger = load_tiger()

# Only load encounter columns needed for ED rate calculation
enc = load_encounters(
    cols=["PatientDurableKey", "IsEdVisit"],
    sample_frac=1.0
)

# --- Attach patient latitude/longitude using CensusBlockGroupFipsCode ---
# This joins:
# patients.CensusBlockGroupFipsCode  <->  tiger.GEOID
# and creates patient centroid columns: pat_lat, pat_lon
pat_geo = attach_patient_coords(pat, tiger)

# --- Define hospital/reference location ---
# Replace these with actual SVH/hospital coordinates if your team has them.
# For now, we use the average centroid of all matched census blocks as a proxy center.
hospital_lat = pat_geo["pat_lat"].mean()
hospital_lon = pat_geo["pat_lon"].mean()

# --- Calculate patient distance from hospital/reference point ---
# haversine_km calculates straight-line distance using lat/lon.
pat_geo["distance_km"] = haversine_km(
    pat_geo["pat_lat"],
    pat_geo["pat_lon"],
    hospital_lat,
    hospital_lon
)

# --- Keep only patient ID + geography + distance ---
# DurableKey links patients to encounters.
pat_distance = pat_geo[
    ["DurableKey", "CensusBlockGroupFipsCode", "distance_km"]
].copy()

# --- Merge encounters with patient distance ---
# Each row in enc is one encounter/visit.
# After merging, each visit now has the patient’s census block and distance.
enc_dist = enc.merge(
    pat_distance,
    left_on="PatientDurableKey",
    right_on="DurableKey",
    how="inner"
)

# --- Compute ED rate by census block group ---
# total_visits = number of encounters from that census block
# total_ed_visits = number of encounters where IsEdVisit == 1
# ed_rate = total_ed_visits / total_visits
distance_ed_df = (
    enc_dist
    .groupby("CensusBlockGroupFipsCode")
    .agg(
        avg_distance_km=("distance_km", "mean"),
        total_visits=("IsEdVisit", "count"),
        total_ed_visits=("IsEdVisit", "sum")
    )
    .reset_index()
)

distance_ed_df["ed_rate"] = (
    distance_ed_df["total_ed_visits"] / distance_ed_df["total_visits"]
)

# Optional cleanup:
# Remove tiny census groups because ED rate can be unstable with very few visits.
distance_ed_df = distance_ed_df[distance_ed_df["total_visits"] >= 20].copy()

# --- Plot scatter ---
plt.figure(figsize=(8, 5))

sns.scatterplot(
    data=distance_ed_df,
    x="avg_distance_km",
    y="ed_rate",
    size="total_visits",
    sizes=(20, 250),
    alpha=0.6,
    legend=False
)

plt.title("Distance vs ED Rate")
plt.xlabel("Average Distance from Reference Location (km)")
plt.ylabel("ED Visit Rate")

save_fig(plt.gcf(), "distance_vs_ed_rate")
plt.show()